In [16]:
from huggingface_hub import hf_hub_download
import zipfile
import pandas as pd

zip_path=hf_hub_download("takala/financial_phrasebank",
                           "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")

def read_phrasebank(agreement="AllAgree"):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(f"FinancialPhraseBank-v1.0/Sentences_{agreement}.txt") as f:
            lines= f.read().decode("latin-1").splitlines()
    rows=[line.rsplit("@",1) for line in lines if "@" in line]
    return pd.DataFrame(rows,columns=["text","label"])

df_all = read_phrasebank("AllAgree")
print(df_all.shape)
df_all.sample(10)

(2264, 2)


,text,label
2183,Alma Media 's operating profit amounted to EUR...,negative
1070,The deal will have no significant effect on th...,neutral
1714,Stora Enso will receive a 19.9 pct equity inte...,neutral
42,"In January-September 2007 , Finnlines ' net sa...",positive
1508,The share capital of Biotie Therapies Corp. co...,neutral
560,Harold W. Young is an independent broker worki...,neutral
2,"In the third quarter of 2010 , net sales incre...",positive
986,"Its customers include local companies Slo Oy ,...",neutral
1849,Total value of the contract is about EUR 10mn .,neutral
1723,The bank 's leasing arm Nordea Liising ended t...,neutral


In [17]:
df_75 = read_phrasebank("75Agree")
print("AllAgree:", df_all.shape, "| 75Agree:", df_75.shape)

AllAgree: (2264, 2) | 75Agree: (3453, 2)


In [18]:
for name ,d in [("AllAgree",df_all) , ("75Agree",df_75)]:
    print(f"---{name}---")
    print(d["label"].value_counts())
    print(d["label"].value_counts(normalize=True).mul(100).round(1))

---AllAgree---
label
neutral     1391
positive     570
negative     303
Name: count, dtype: int64
label
neutral     61.4
positive    25.2
negative    13.4
Name: proportion, dtype: float64
---75Agree---
label
neutral     2146
positive     887
negative     420
Name: count, dtype: int64
label
neutral     62.1
positive    25.7
negative    12.2
Name: proportion, dtype: float64


In [19]:
print(df_75["text"].str.split().str.len().describe())
print("Labels:", df_75["label"].unique())

count    3453.000000
mean       22.756733
std        10.067928
min         2.000000
25%        15.000000
50%        21.000000
75%        29.000000
max        81.000000
Name: text, dtype: float64
Labels: <ArrowStringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str


# PhraseBank Findings

| tier | rows | neutral | positive | negative |
|------|-----:|---------:|----------:|----------:|
| AllAgree | 2,264 | 61.4% | 25.2% | 13.4% |
| 75Agree  | 3,453 | 62.1% | 25.7% | 12.2% |

### Sentence lengths

- **Median:** 21 words
- **Maximum:** 81 words

### Training decision

We will train on **75Agree** because **it contains more training examples (3,453 vs. 2,264), allowing the model to learn better while still maintaining high-quality labels (at least 75% annotator agreement).**

### Warning for Day 12

A model that always predicts **"neutral"** scores approximately **62%** accuracy.

Therefore, **accuracy alone is not a reliable evaluation metric**.

Instead, we will use **per-class F1 score** to evaluate the model.

The **rarest class** is **negative (~12%)**, making it the most important class to classify correctly.

In [20]:
import kagglehub
from pathlib import Path

path=Path(kagglehub.dataset_download("miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests"))
print(path)
list(path.rglob("*.csv"))

C:\Users\Dell\.cache\kagglehub\datasets\miguelaenlle\massive-stock-news-analysis-db-for-nlpbacktests\versions\2


[WindowsPath('C:/Users/Dell/.cache/kagglehub/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests/versions/2/analyst_ratings_processed.csv'),
 WindowsPath('C:/Users/Dell/.cache/kagglehub/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests/versions/2/raw_analyst_ratings.csv'),
 WindowsPath('C:/Users/Dell/.cache/kagglehub/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests/versions/2/raw_partner_headlines.csv')]

In [21]:
news= pd.read_csv(path / "raw_analyst_ratings.csv")
print(news.shape)
print(news.columns.tolist())
news.head()

(1407328, 6)
['Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock']


,Unnamed: 0,headline,url,publisher,date,stock
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [22]:
print(news["stock"].isin(["AAPL","MSFT","GOOGL","AMZN","NVDA","TSLA","META","JPM","JNJ","XOM"]).sum(), "rows for our 10")
print(news["date"].head(3))

10841 rows for our 10
0    2020-06-05 10:30:54-04:00
1    2020-06-03 10:45:20-04:00
2    2020-05-26 04:30:07-04:00
Name: date, dtype: str


In [23]:
probe = news[news["stock"].isin(
    ["AAPL","MSFT","GOOGL","AMZN","NVDA","TSLA","META","JPM","JNJ","XOM","FB"])]
print(probe["stock"].value_counts())

stock
NVDA     3146
JNJ      2928
TSLA     1875
GOOGL    1579
XOM       584
AAPL      441
FB        380
AMZN      278
JPM        10
Name: count, dtype: int64


In [24]:
partner = pd.read_csv(path / "raw_partner_headlines.csv")
print(partner.shape)
print(partner.columns.tolist())
print(partner[partner["stock"].isin(
    ["AAPL","MSFT","GOOGL","AMZN","NVDA","TSLA","META","JPM","JNJ","XOM","FB"])]["stock"].value_counts())

(1845559, 6)
['Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock']
stock
JPM      2873
GOOGL     175
FB         53
AAPL       32
XOM         5
Name: count, dtype: int64


## Known data limitation — Microsoft (MSFT)

The news dataset (scraped from Benzinga, ~2009–2020) contains **zero headlines
tagged MSFT** in any of its files — verified by direct search of both
`raw_analyst_ratings.csv` and `raw_partner_headlines.csv`.

**Consequences:**
- MSFT will have no news-sentiment features in Week 2 models
- The dashboard must handle a ticker with no news gracefully ("no news data available")

**Decision:** accepted and documented rather than patched. Supplementing MSFT
from a second news source is on the bonus list, not in scope for the core build.

Related fix in the same pipeline: Meta traded as ticker **FB** until 2022, so the
loader translates FB → META before filtering — otherwise Meta would have
appeared to have no news either.